In [14]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parents[3]))  # .../SystemSimulation
from demos.ControlledPendulum.src.master_pendulum.components import (
    OpenSimPendulum,
    FEMPendulum,
    PendulumODE, 
    constant_torque,
    ramp_torque,
    zero_torque,
)
from demos.ControlledPendulum.src.master_pendulum import MasterPendulum
import demos.ControlledPendulum.src.master_pendulum.components.fem.pendulum_config as config
from thesis.plot_setup import set_professional_style
plt = set_professional_style()

from syssimx import Connection, System
from syssimx.system.connection import EventConnection
from syssimx.viz.system_graph_visualizer import SystemGraphVisualizer

In [16]:
mesh_params = config.MeshParameters()

init_params = config.InitialConditionParameters()
init_params.angular_position_deg = np.rad2deg(0.3) # Initial angle
init_params.angular_velocity = 1
q0 = np.deg2rad(init_params.angular_position_deg)
init_params.drive_torque = 0 # Nm

mat_params = config.MaterialParameters()
mat_params.E_pendulum = 2.1e11  # Young's modulus for the pendulum
mat_params.nu_pendulum = 0.3    # Poisson's ratio for the pendulum
mat_params.rho_pendulum = 7800  # Density for the pendulum

sim_params = config.SimulationParameters()
sim_params.tau = 0.01
sim_params.t_end = 1
sim_params.with_contact = True
sim_params.use_gravity = True

contact_params = config.ContactParameters()
contact_params.kn = 1e10

anim_params = config.AnimationParameters()
anim_params.animate = True

fem_parameters = {
    'mat_params': mat_params,
    'contact_params': contact_params,
    'init_params': init_params,
    'sim_params': sim_params,
    'anim_params': anim_params,
    'mesh_params': mesh_params,
}

pendulum = MasterPendulum(name="MasterPendulum", initial_mode="FMU")

pendulum.set_parameters(**{"FEM": fem_parameters})
pendulum.initialize(t0=0.0)

t = 0.0
dt = pendulum.fem.sim_params.tau
t_end = pendulum.fem.sim_params.t_end

In [18]:
for model in pendulum.models.values():
    outputs = model.get_outputs()
    print(model.name)
    keys = list(sorted(outputs.keys()))
    for key in keys:
        print(f"  {key:>5}: {outputs[key]:.4f} ")

FEM_Pendulum
  alpha: -3.8733 radian / second ** 2 
  omega: 1.0000 radian / second 
      q: 0.3000 radian 
OpenSim_Pendulum
  alpha: -3.8733 radian / second ** 2 
  omega: 1.0000 radian / second 
      q: 0.3000 radian 
FMU_Pendulum
  alpha: -3.8733 radian / second ** 2 
  omega: 1.0000 radian / second 
      q: 0.3000 radian 


In [19]:
pendulum.hysteresis.dwell_time

0.05

In [20]:
pendulum.do_step(0, 0.1)

In [6]:
state = pendulum.get_state()
state

{'alpha': {'unit': 'rad/s2', 'value': -4.846874603109723},
 'omega': {'unit': 'rad/s', 'value': 0.5592723472753408},
 'q': {'unit': 'rad', 'value': 0.37879835602874723},
 'torque': {'unit': 'N.m', 'value': 0.0}}

In [21]:
proposed_mode = 'FEM'

pendulum._switch_mode(proposed_mode, 0.1)


[MasterPendulum] Switching: FMU to FEM @ t=0.1000s


In [22]:
pendulum.get_state()

{'q': {'value': np.float64(0.3787983560287471), 'unit': 'rad'},
 'omega': {'value': 0.559272347275342, 'unit': 'rad/s'},
 'alpha': {'value': -4.846874603109724, 'unit': 'rad/s**2'},
 'torque': {'value': 0.0, 'unit': 'N*m'},
 'gap': {'value': 0.29584342736127534, 'unit': 'm'},
 'gap_prev': {'value': inf, 'unit': 'm'}}

In [23]:
t_fmu, q_fmu = pendulum.fmu.get_history_arrays('q')
t_fem, q_fem = pendulum.fem.get_history_arrays('q')

In [24]:
t_fmu, q_fmu

(array([0. , 0.1]), {'q': array([0.3       , 0.37879836])})

In [25]:
t_fem, q_fem

(array([0.]), {'q': array([0.3])})